# SETUP

In [1]:
import os
os.chdir('/root/eallmancy')

from eallmancy.engine.llm_solution_engine_v3 import EngineV3
from eallmancy import Artifact
from eallmancy import SinglePartSolution, ProblemSpec
from eallmancy.engine.engine_trace import EngineTrace
from collections import defaultdict
import datetime

import os
import re

# TODO: THIS IS ULTRA HACKY, PUT API SETTINGS IN CONTROL PANEL
with open('/root/openai_api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().replace('\n','')
with open('/root/gemini_api_key') as f:
    os.environ['GEMINI_API_KEY'] = f.read().replace('\n','')


# from experiments.scievo import SciEvo
from experiments.toyeoh import ToyEoH
from problems.llm4ad import (    
    CarMountainProblemSpec,
    PendulumProblemSpec,
    
    StressTrainProblemSpec,
    BactGrowProblemSpec,
    
    AdmissibleSetProblemSpec,
    VRPTWProblemSpec
)

# sim func

In [2]:
CHATGPT_4OMINI_API = dict(
            host="api.openai.com",
            key=os.environ.get('OPENAI_API_KEY'),
            model="gpt-4o-mini",
            timeout=20
        )

GEMINI_25FLASH_API = dict(
            base_url='https://generativelanguage.googleapis.com/v1beta',
            api_key=os.environ.get('GEMINI_API_KEY'),
            model='gemini-2.0-flash',
            timeout=20,
            openai_format=True
        )

In [3]:

async def nowhistles_seoh_ps(problem_spec, api, max_samples=200,):
    engine = EngineV3()
    eallm = ToyEoH(
        # engine=engine,
        api=api,
        ps=problem_spec,
        config=dict(max_samples=max_samples)
    )
    
    best_soln, trace = await eallm.run()
    return trace


async def uniformselection_seoh_ps(problem_spec, api, max_samples=200,):
    engine = EngineV3()
    eallm = ToyEoH(
        # engine=engine,
        api=api,
        ps=problem_spec,
        config=dict(max_samples=max_samples, unbiased_selection=True)
    )
    
    best_soln, trace = await eallm.run()
    return trace


# sim running

In [30]:
llms_to_run = {
    # '4omini': CHATGPT_4OMINI_API,
    'gemini25f': GEMINI_25FLASH_API
}

ps_to_test = [

    CarMountainProblemSpec(maximization=True),
    PendulumProblemSpec(maximization=True),
    
    StressTrainProblemSpec(maximization=True),
    BactGrowProblemSpec(maximization=True),
    
    ## AdmissibleSetProblemSpec(maximization=True, dimension=21, weight=15),
    ## AdmissibleSetProblemSpec(maximization=True, dimension=12, weight=7),
    AdmissibleSetProblemSpec(maximization=True, dimension=15, weight=10),
    VRPTWProblemSpec(maximization=True)
]


res = defaultdict(lambda : [])

for llm_name, llm_api in llms_to_run.items():
    for idx, ps in enumerate(ps_to_test):
        trace = await uniformselection_seoh_ps(ps, api=llm_api, max_samples=200)
        res[ps.__class__.__name__].append(trace)
        time = datetime.datetime.now().strftime('%Y%m%d-%H%S')
        print(f'[{time}]', llm_name, ps.__class__.__name__, max([a.fitness for a in trace.list]))
        
        trace.save(f'/root/eallmancy/experiments/toyeoh/sims/{llm_name}_{ps.__class__.__name__}_UNIFORM-{time}.pk')
    

[20250629-1327] gemini25f CarMountainProblemSpec -0.236
[20250629-1344] gemini25f PendulumProblemSpec -1.0000000000000018


<string>:21: RuntimeWarning: invalid value encountered in power
<string>:22: RuntimeWarning: invalid value encountered in divide
<string>:23: RuntimeWarning: invalid value encountered in divide
<string>:23: RuntimeWarning: divide by zero encountered in divide
<string>:17: RuntimeWarning: invalid value encountered in power
<string>:23: RuntimeWarning: invalid value encountered in divide
<string>:23: RuntimeWarning: divide by zero encountered in divide
<string>:29: RuntimeWarning: invalid value encountered in power
<string>:23: RuntimeWarning: invalid value encountered in divide
<string>:23: RuntimeWarning: invalid value encountered in power
<string>:26: RuntimeWarning: invalid value encountered in divide
<string>:26: RuntimeWarning: divide by zero encountered in divide
<string>:32: RuntimeWarning: invalid value encountered in scalar multiply
<string>:16: RuntimeWarning: invalid value encountered in divide
<string>:16: RuntimeWarning: divide by zero encountered in divide
<string>:16: Run

[20250629-1457] gemini25f StressTrainProblemSpec -0.0016926370916203238


<string>:28: RuntimeWarning: divide by zero encountered in divide
<string>:33: RuntimeWarning: divide by zero encountered in divide
<string>:28: RuntimeWarning: invalid value encountered in power
<string>:33: RuntimeWarning: invalid value encountered in power
<string>:31: RuntimeWarning: divide by zero encountered in divide
<string>:26: RuntimeWarning: divide by zero encountered in divide
<string>:30: RuntimeWarning: divide by zero encountered in divide
<string>:27: RuntimeWarning: divide by zero encountered in divide
<string>:31: RuntimeWarning: divide by zero encountered in divide
<string>:30: RuntimeWarning: divide by zero encountered in divide
<string>:35: RuntimeWarning: divide by zero encountered in divide
<string>:29: RuntimeWarning: divide by zero encountered in divide
<string>:31: RuntimeWarning: invalid value encountered in divide
<string>:29: RuntimeWarning: divide by zero encountered in divide
<string>:35: RuntimeWarning: divide by zero encountered in divide
<string>:23: Ru

[20250629-1426] gemini25f BactGrowProblemSpec -0.014703807702108154


<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encountered in scalar divide
<string>:36: RuntimeWarning: divide by zero encount

[20250629-1437] gemini25f AdmissibleSetProblemSpec 1941


<string>:42: RuntimeWarning: invalid value encountered in divide
<string>:53: RuntimeWarning: invalid value encountered in divide
<string>:71: RuntimeWarning: invalid value encountered in subtract
<string>:57: RuntimeWarning: invalid value encountered in divide
<string>:46: RuntimeWarning: invalid value encountered in divide
<string>:62: RuntimeWarning: invalid value encountered in divide
<string>:78: RuntimeWarning: invalid value encountered in divide


[20250629-1543] gemini25f VRPTWProblemSpec -21.18053669115129


In [5]:
import datetime

print(datetime.datetime.now())

2025-06-28 19:38:45.328147


In [20]:
!pwd

/root/eallmancy


In [28]:
!pwd

/root/eallmancy/experiments


In [29]:
!ls toyeoh/sims


4omini_AdmissibleSetProblemSpec_20250627-0921.pk
4omini_AdmissibleSetProblemSpec_20250627-1852.pk
4omini_AdmissibleSetProblemSpec_UNIFORM-20250627-2102.pk
4omini_AdmissibleSetProblemSpec_UNIFORM-20250627-2351.pk
4omini_BactGrowProblemSpec_20250627-0944.pk
4omini_BactGrowProblemSpec_20250627-1800.pk
4omini_BactGrowProblemSpec_UNIFORM-20250627-2105.pk
4omini_BactGrowProblemSpec_UNIFORM-20250627-2303.pk
4omini_CarMountainProblemSpec_20250626-2257.pk
4omini_CarMountainProblemSpec_20250627-1745.pk
4omini_PendulumProblemSpec_20250627-1355.pk
4omini_PendulumProblemSpec_20250627-1749.pk
4omini_PendulumProblemSpec_UNIFORM-20250627-2005.pk
4omini_PendulumProblemSpec_UNIFORM-20250627-2216.pk
4omini_StressTrainProblemSpec_20250627-0709.pk
4omini_StressTrainProblemSpec_20250627-1745.pk
4omini_StressTrainProblemSpec_UNIFORM-20250627-2010.pk
4omini_StressTrainProblemSpec_UNIFORM-20250627-2313.pk
4omini_VRPTWProblemSpec_20250627-1023.pk
4omini_VRPTWProblemSpec_20250627-1933.pk
4omini_VRPTWProblemSpec_

### scratch